# EZhire Metrics Comparison
Loads saved models, computes scores, and writes ensemble artifacts for the Gradio app.

In [ ]:
!pip install -q sentence-transformers datasets scikit-learn nltk PyMuPDF einops accelerate

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import nltk
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, ndcg_score, r2_score
from sklearn.model_selection import StratifiedKFold
from scipy.stats import spearmanr, pearsonr
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from IPython.display import display

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = os.path.abspath(".")
MODEL_DIR = os.path.join(BASE_DIR, "saved_models")
ARTIFACT_DIR = os.path.join(BASE_DIR, "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

STOP_WORDS = set(stopwords.words("english"))
print(f"Device: {DEVICE}")

In [ ]:
ATS_MIN, ATS_MAX = 18.3, 90.7

def split_sep(text):
    if not isinstance(text, str): text = str(text)
    if '[SEP]' in text:
        a, b = text.split('[SEP]', 1)
        return a.strip(), b.strip()
    mid = len(text) // 2
    return text[:mid].strip(), text[mid:].strip()

def raw_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [re.sub(r'[ \t\f\v]+', ' ', l).strip() for l in text.split('\n')]
    return ' '.join(l for l in lines if l)

def clean_text(text):
    if not isinstance(text, str): text = str(text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())
    tokens = word_tokenize(re.sub(r'\s+', ' ', text).strip())
    return ' '.join(t for t in tokens if t not in STOP_WORDS and len(t) > 1)

def normalize_score(series):
    v = pd.to_numeric(series, errors='coerce').astype(float)
    return ((v - ATS_MIN) / (ATS_MAX - ATS_MIN)).clip(0, 1)

def denormalize_score(v):
    return np.asarray(v, float) * (ATS_MAX - ATS_MIN) + ATS_MIN

def build_df(src):
    splits = src['text'].apply(split_sep)
    ats = pd.to_numeric(src['ats_score'], errors='coerce').fillna(ATS_MIN)
    return pd.DataFrame({
        'resume_raw': splits.apply(lambda x: raw_text(x[0])),
        'jd_raw': splits.apply(lambda x: raw_text(x[1])),
        'resume_clean': splits.apply(lambda x: clean_text(x[0])),
        'jd_clean': splits.apply(lambda x: clean_text(x[1])),
        'original_label': src['original_label'].values,
        'ats_score_raw': ats.values,
        'ground_truth': normalize_score(ats).values
    }).dropna(subset=['resume_raw', 'jd_raw']).reset_index(drop=True)

ds = load_dataset('0xnbk/resume-ats-score-v1-en')
df_train = ds['train'].to_pandas()
df_val = ds['validation'].to_pandas()

df_tr = build_df(df_train)
df_vl = build_df(df_val)
print(f'Train: {len(df_tr)} | Val: {len(df_vl)}')
print(f'GT range train: {df_tr["ground_truth"].min():.3f} - {df_tr["ground_truth"].max():.3f}')

## Shared Chunking Infrastructure

In [ ]:
CHUNK_OVERLAP = 38
MAX_RESUME_CHUNKS = 10
MAX_JD_CHUNKS = 8
TOP_K_CHUNK_PAIRS = 5
ENCODE_BATCH = 16 if DEVICE == 'cuda' else 8

def extract_first_sentence(text, max_chars=150):
    t = raw_text(text)
    m = re.search(r'(?<=[a-zA-Z0-9])[.!?]', t)
    if m and m.start() > 10:
        sentence = t[:m.start() + 1].strip()
    else:
        sentence = t[:max_chars].strip()
    return sentence[:max_chars]

def token_chunk_body(body, tokenizer, max_tokens=512, overlap=CHUNK_OVERLAP, prefix_tokens=0):
    body = raw_text(body)
    if not body:
        return []
    ids = tokenizer.encode(body, add_special_tokens=False, truncation=False)
    effective_max = max(64, max_tokens - prefix_tokens - 2)
    if len(ids) <= effective_max:
        return [body]
    overlap = min(overlap, effective_max // 2)
    step = effective_max - overlap
    chunks = []
    for start in range(0, len(ids), step):
        piece = ids[start:start + effective_max]
        chunk = raw_text(tokenizer.decode(piece, skip_special_tokens=True))
        if chunk:
            chunks.append(chunk)
        if start + effective_max >= len(ids):
            break
    return chunks

def make_text_chunks(text, tokenizer, model_max_tokens=512, overlap=CHUNK_OVERLAP,
                     max_chunks=MAX_RESUME_CHUNKS, source_label='DOCUMENT'):
    text = raw_text(text)
    if not text:
        return []
    doc_ctx = extract_first_sentence(text)
    prefix = f'[DOC]: {doc_ctx} [{source_label}] ' if doc_ctx else f'[{source_label}] '
    prefix_tokens = len(tokenizer.encode(prefix, add_special_tokens=False))
    body_chunks = token_chunk_body(
        text, tokenizer,
        max_tokens=model_max_tokens,
        overlap=overlap,
        prefix_tokens=prefix_tokens
    )
    chunks = [f'{prefix}{c}' for c in body_chunks]
    if len(chunks) > max_chunks:
        keep = np.linspace(0, len(chunks) - 1, max_chunks, dtype=int).tolist()
        chunks = [chunks[i] for i in keep]
    fallback = text[:2000]
    return chunks or [f'{prefix}{fallback}']

def aggregate_chunk_sims(sims, top_k=TOP_K_CHUNK_PAIRS):
    sims = np.asarray(sims, float)
    if sims.size == 0: return 0.0
    flat = sims.reshape(-1)
    k = min(top_k, len(flat))
    top_mean = float(np.partition(flat, -k)[-k:].mean())
    coverage = float((sims.max(axis=0).mean() + sims.max(axis=1).mean()) / 2)
    return float(np.clip(0.75 * top_mean + 0.25 * coverage, 0.0, 1.0))

def score_doc_pair(model, t1, t2, left='RESUME', right='JOB'):
    tok = model.tokenizer
    mlen = model.max_seq_length
    c1 = make_text_chunks(t1, tok, model_max_tokens=mlen,
                          max_chunks=MAX_RESUME_CHUNKS, source_label=left)
    c2 = make_text_chunks(t2, tok, model_max_tokens=mlen,
                          max_chunks=MAX_JD_CHUNKS, source_label=right)
    e1 = model.encode(c1, convert_to_tensor=True, normalize_embeddings=True,
                      batch_size=ENCODE_BATCH, show_progress_bar=False)
    e2 = model.encode(c2, convert_to_tensor=True, normalize_embeddings=True,
                      batch_size=ENCODE_BATCH, show_progress_bar=False)
    sims = util.cos_sim(e1, e2).detach().cpu().numpy()
    return aggregate_chunk_sims(sims)

## Load Saved Models

In [ ]:
def require_model(path, label):
    if not os.path.exists(os.path.join(path, 'modules.json')):
        raise FileNotFoundError(f'Missing {label} at {path}. Run the training notebook first.')

MPNET_PATH = os.path.join(MODEL_DIR, 'ezhire-mpnet')
ROBERTA_PATH = os.path.join(MODEL_DIR, 'ezhire-roberta')
JINA_PATH = os.path.join(MODEL_DIR, 'ezhire-jina')

require_model(MPNET_PATH, 'mpnet')
require_model(ROBERTA_PATH, 'roberta')
require_model(JINA_PATH, 'jina')

mpnet_model = SentenceTransformer(MPNET_PATH, device=DEVICE)
mpnet_model.max_seq_length = 384

roberta_model = SentenceTransformer(ROBERTA_PATH, device=DEVICE)
roberta_model.max_seq_length = 384

jina_model = SentenceTransformer(JINA_PATH, trust_remote_code=True, device=DEVICE)
jina_model.max_seq_length = 8192

print('Models loaded successfully')

In [ ]:
print('Evaluating Model A (mpnet) on full validation set...')
mpnet_preds = []
for i, row in df_vl.iterrows():
    mpnet_preds.append(score_doc_pair(mpnet_model, row['resume_raw'], row['jd_raw']))
    if (i + 1) % 50 == 0: print(f'  {i + 1}/{len(df_vl)}')
df_vl['mpnet_score'] = [round(s * 100, 2) for s in mpnet_preds]
yt = df_vl['ground_truth'].values
yp = df_vl['mpnet_score'].values
mpnet_pearson = float(pearsonr(yp / 100, yt)[0])
mpnet_spearman = float(spearmanr(yp / 100, yt).correlation)
mpnet_mae = float(mean_absolute_error(yt, yp / 100))
mpnet_rmse = float(np.sqrt(np.mean((denormalize_score(yt) - denormalize_score(yp / 100)) ** 2)))
mpnet_r2 = float(r2_score(yt, yp / 100))

print('Evaluating Model B (roberta) on full validation set...')
roberta_preds = []
for i, row in df_vl.iterrows():
    roberta_preds.append(score_doc_pair(roberta_model, row['resume_raw'], row['jd_raw']))
    if (i + 1) % 50 == 0: print(f'  {i + 1}/{len(df_vl)}')
df_vl['roberta_score'] = [round(s * 100, 2) for s in roberta_preds]
yp = df_vl['roberta_score'].values
roberta_pearson = float(pearsonr(yp / 100, yt)[0])
roberta_spearman = float(spearmanr(yp / 100, yt).correlation)
roberta_mae = float(mean_absolute_error(yt, yp / 100))
roberta_rmse = float(np.sqrt(np.mean((denormalize_score(yt) - denormalize_score(yp / 100)) ** 2)))
roberta_r2 = float(r2_score(yt, yp / 100))

print('Evaluating Model C (Jina) on full validation set...')
jina_preds = []
for i, row in df_vl.iterrows():
    e1 = jina_model.encode(row['resume_raw'], convert_to_tensor=True,
                           normalize_embeddings=True, show_progress_bar=False)
    e2 = jina_model.encode(row['jd_raw'], convert_to_tensor=True,
                           normalize_embeddings=True, show_progress_bar=False)
    jina_preds.append(float(util.cos_sim(e1.unsqueeze(0), e2.unsqueeze(0))[0][0]))
    if (i + 1) % 50 == 0: print(f'  {i + 1}/{len(df_vl)}')
df_vl['jina_score'] = [round(s * 100, 2) for s in jina_preds]
yp = df_vl['jina_score'].values
jina_pearson = float(pearsonr(yp / 100, yt)[0])
jina_spearman = float(spearmanr(yp / 100, yt).correlation)
jina_mae = float(mean_absolute_error(yt, yp / 100))
jina_rmse = float(np.sqrt(np.mean((denormalize_score(yt) - denormalize_score(yp / 100)) ** 2)))
jina_r2 = float(r2_score(yt, yp / 100))

print('Model A (mpnet) Results:')
print(f'  Pearson  : {mpnet_pearson:+.4f} | Spearman: {mpnet_spearman:+.4f} | MAE: {mpnet_mae:.4f}')
print('Model B (roberta) Results:')
print(f'  Pearson  : {roberta_pearson:+.4f} | Spearman: {roberta_spearman:+.4f} | MAE: {roberta_mae:.4f}')
print('Model C (jina) Results:')
print(f'  Pearson  : {jina_pearson:+.4f} | Spearman: {jina_spearman:+.4f} | MAE: {jina_mae:.4f}')

In [ ]:
print('Comparing Model A vs Model B...')
print(f'  mpnet   Pearson: {mpnet_pearson:+.4f}')
print(f'  roberta Pearson: {roberta_pearson:+.4f}')

if mpnet_pearson >= roberta_pearson:
    best_sbert_name = 'mpnet'
    best_sbert_score_col = 'mpnet_score'
    print(f'Best SBERT -> Model A (mpnet) | Pearson={mpnet_pearson:+.4f}')
else:
    best_sbert_name = 'roberta'
    best_sbert_score_col = 'roberta_score'
    print(f'Best SBERT -> Model B (roberta) | Pearson={roberta_pearson:+.4f}')

print('This model will be used as the SBERT component in the final ensemble.')

In [ ]:
def tfidf_score(t1, t2):
    try:
        m = TfidfVectorizer().fit_transform([t1, t2])
        return float(cosine_similarity(m[0:1], m[1:2])[0][0])
    except Exception:
        return 0.0

print('Computing TF-IDF scores...')
tfidf_preds = []
for i, row in df_vl.iterrows():
    tfidf_preds.append(round(tfidf_score(row['resume_clean'], row['jd_clean']) * 100, 2))
    if (i + 1) % 100 == 0: print(f'  {i + 1}/{len(df_vl)}')
df_vl['tfidf_score'] = tfidf_preds

print(f'Grid searching ensemble weight for best SBERT ({best_sbert_name})...')
best_sbert_preds = df_vl[best_sbert_score_col].values
tfidf_arr = df_vl['tfidf_score'].values
yt = df_vl['ground_truth'].values

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tiers_for_cv = df_vl['original_label'].values
weight_results = {}

for w in np.round(np.arange(0.50, 1.01, 0.05), 2):
    fold_scores = []
    for _, val_idx in kf.split(np.zeros(len(df_vl)), tiers_for_cv):
        ens = w * best_sbert_preds[val_idx] + (1 - w) * tfidf_arr[val_idx]
        sp = spearmanr(ens, yt[val_idx]).correlation
        fold_scores.append(sp)
    weight_results[w] = round(float(np.mean(fold_scores)), 4)

BEST_W = max(weight_results, key=weight_results.get)
print('CV Weight Grid:')
for w, s in sorted(weight_results.items()):
    marker = ' <-- BEST' if w == BEST_W else ''
    print(f'  SBERT weight={w:.2f}  Avg Spearman={s:.4f}{marker}')
print(f'Optimal SBERT weight: {BEST_W:.2f} | TF-IDF weight: {1 - BEST_W:.2f}')

def ensemble_score(s, t, sw=BEST_W):
    return round((sw * s + (1 - sw) * t) * 100, 2)

def get_tier(score):
    if score >= 70: return 'Strong Match'
    if score >= 45: return 'Potential Fit'
    return 'Poor Match'

df_vl['ensemble_score'] = [
    round(BEST_W * s + (1 - BEST_W) * t, 2)
    for s, t in zip(best_sbert_preds, tfidf_arr)
]
df_vl['tier'] = df_vl['ensemble_score'].apply(get_tier)
df_sample = df_vl.copy()
print('Ensemble scoring complete.')

In [ ]:
def safe_r(x, y, kind='pearson'):
    if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0: return 0.0
    v = pearsonr(x, y)[0] if kind == 'pearson' else spearmanr(x, y).correlation
    return 0.0 if np.isnan(v) else float(v)

def precision_at_k(yt, yp, k=5, thresh=0.5):
    top = sorted(zip(yp, yt), reverse=True)[:k]
    return sum(1 for _, t in top if t >= thresh) / k

def mrr_score(yt, yp, thresh=0.5):
    for rank, (_, t) in enumerate(sorted(zip(yp, yt), reverse=True), 1):
        if t >= thresh: return 1.0 / rank
    return 0.0

def full_metrics(yt_norm, yp_pct, name, yt_raw=None):
    yt = np.asarray(yt_norm, float)
    yp = np.asarray(yp_pct, float)
    ypr = np.clip(yp / 100, 0, 1)
    ytr = denormalize_score(yt) if yt_raw is None else np.asarray(yt_raw, float)
    ydp = denormalize_score(ypr)
    mask = ~np.isnan(yt) & ~np.isnan(yp)
    yt, yp, ypr, ytr, ydp = yt[mask], yp[mask], ypr[mask], ytr[mask], ydp[mask]
    if len(yt) < 2: return {}
    sp = safe_r(ypr, yt, 'spearman')
    pe = safe_r(ypr, yt, 'pearson')
    mae = mean_absolute_error(yt, ypr)
    maer = mean_absolute_error(ytr, ydp)
    rmse = float(np.sqrt(np.mean((ytr - ydp) ** 2)))
    r2 = float(r2_score(yt, ypr))
    nd = ndcg_score([yt], [yp])
    p5 = precision_at_k(yt, ypr, 5)
    p10 = precision_at_k(yt, ypr, 10)
    m = mrr_score(yt, ypr)
    print(f'{name:18s} Spearman={sp:+.4f} Pearson={pe:+.4f} '
          f'MAE={mae:.4f} MAE_raw={maer:.2f} RMSE={rmse:.2f} '
          f'R2={r2:+.4f} NDCG={nd:.4f} P@5={p5:.2f} MRR={m:.4f}')
    return dict(model=name, spearman=sp, pearson=pe, mae=mae,
                mae_raw=maer, rmse_raw=rmse, r2_raw=r2, ndcg=nd,
                precision_at_5=p5, precision_at_10=p10, mrr=m)

print('Full metrics on validation set:')
yt = df_sample['ground_truth'].values
yt_r = df_sample['ats_score_raw'].values
r_mpnet = full_metrics(yt, df_sample['mpnet_score'].values, 'mpnet', yt_r)
r_roberta = full_metrics(yt, df_sample['roberta_score'].values, 'roberta', yt_r)
r_jina = full_metrics(yt, df_sample['jina_score'].values, 'jina', yt_r)
r_tfidf = full_metrics(yt, df_sample['tfidf_score'].values, 'TF-IDF', yt_r)
r_ensemble = full_metrics(yt, df_sample['ensemble_score'].values, f'Ensemble({best_sbert_name}+TF-IDF)', yt_r)

metrics_df = pd.DataFrame([r_mpnet, r_roberta, r_jina, r_tfidf, r_ensemble])
if 'model' in metrics_df.columns:
    metrics_df = metrics_df.set_index('model')
print('Summary Table:')
display(metrics_df.round(4))

In [ ]:
metrics_path = os.path.join(ARTIFACT_DIR, 'metrics_df.csv')
scores_path = os.path.join(ARTIFACT_DIR, 'validation_scores.csv')
config_path = os.path.join(ARTIFACT_DIR, 'ensemble_config.json')

metrics_df.reset_index().to_csv(metrics_path, index=False)
df_sample[[
]].to_csv(scores_path, index=False)

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump({
        'best_sbert_name': best_sbert_name,
        'best_sbert_weight': float(BEST_W)
    }, f, indent=2)

print(f'Wrote metrics to: {metrics_path}')
print(f'Wrote validation scores to: {scores_path}')
print(f'Wrote ensemble config to: {config_path}')